# EDA Silver Indicateur 2

In [ ]:
from pathlib import Path

import folium
import pandas as pd
from folium.plugins import MarkerCluster
from shapely import wkb
from shapely.geometry import mapping

def resolve_silver_dir() -> Path:
    candidates = [
        Path.cwd() / 'silver',
        Path.cwd() / 'Indicateur_2' / 'silver',
        Path.cwd().parent / 'Indicateur_2' / 'silver',
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir() and any(candidate.iterdir()):
            return candidate
    raise FileNotFoundError('Impossible de localiser le dossier silver de lindicateur 2')

SILVER_DIR = resolve_silver_dir()
PARIS_CENTER = [48.8566, 2.3522]
PARIS_ZOOM = 11

def create_base_map(title: str):
    m = folium.Map(location=PARIS_CENTER, zoom_start=PARIS_ZOOM, tiles='CartoDB positron')
    folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
    folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)
    return m

def add_point_layer(m, df, lat_col='latitude', lon_col='longitude', layer_name='Points', color='#1f77b4'):
    if lat_col not in df.columns or lon_col not in df.columns:
        return
    point_df = df.dropna(subset=[lat_col, lon_col]).copy()
    if point_df.empty:
        return
    point_df[lat_col] = pd.to_numeric(point_df[lat_col], errors='coerce')
    point_df[lon_col] = pd.to_numeric(point_df[lon_col], errors='coerce')
    point_df = point_df.dropna(subset=[lat_col, lon_col])

    cluster = MarkerCluster(name=layer_name).add_to(m)
    for _, row in point_df.iterrows():
        tooltip_parts = []
        for column in df.columns:
            if column not in ['geo_shape', 'geo_point_2d', 'latitude', 'longitude', 'x_wgs84', 'y_wgs84']:
                if column in row and pd.notna(row[column]):
                    tooltip_parts.append(f"{column}: {row[column]}")
        tooltip = ' | '.join(tooltip_parts) if tooltip_parts else layer_name
        folium.CircleMarker(
            location=[float(row[lat_col]), float(row[lon_col])],
            radius=3,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            tooltip=tooltip,
        ).add_to(cluster)

def add_shape_layer(m, df, geom_col='geo_shape', layer_name='Zones', color='#f28e2b'):
    if geom_col not in df.columns:
        return

    features = []
    for _, row in df.iterrows():
        value = row.get(geom_col)
        if value is None or pd.isna(value):
            continue
        try:
            geometry = wkb.loads(bytes(value))
        except Exception:
            continue

        properties = {}
        for column in df.columns:
            if column not in ['geo_shape', 'geo_point_2d', 'latitude', 'longitude']:
                if column in row and pd.notna(row[column]):
                    properties[column] = str(row[column])

        features.append({
            'type': 'Feature',
            'geometry': mapping(geometry),
            'properties': properties,
        })

    if not features:
        return

    folium.GeoJson(
        {'type': 'FeatureCollection', 'features': features},
        name=layer_name,
        style_function=lambda feature: {
            'color': color,
            'weight': 2,
            'fillColor': color,
            'fillOpacity': 0.25,
        },
        tooltip=folium.GeoJsonTooltip(fields=list(features[0]['properties'].keys())) if features and features[0]['properties'] else None,
    ).add_to(m)

def finalize_map(m):
    folium.LayerControl(collapsed=False).add_to(m)
    return m



In [ ]:
df = pd.read_parquet(SILVER_DIR / "ilots-de-fraicheur-espaces-verts-frais.parquet")
m = create_base_map("ilots-de-fraicheur-espaces-verts-frais.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / "les-arbres.parquet")
m = create_base_map("les-arbres.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / "lieux-de-tournage-a-paris.parquet")
m = create_base_map("lieux-de-tournage-a-paris.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / "liste_des_associations_parisiennes.parquet")
m = create_base_map("liste_des_associations_parisiennes.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / "plan de voirie.parquet")
m = create_base_map("plan de voirie.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / "que-faire-a-paris.parquet")
m = create_base_map("que-faire-a-paris.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / "zones-touristiques-internationales.parquet")
m = create_base_map("zones-touristiques-internationales.parquet")
add_point_layer(m, df, layer_name='Points', color='#1f77b4')
add_shape_layer(m, df, layer_name='Zones', color='#f28e2b')
finalize_map(m)
m